# 1. Merge WSC

In [197]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.strtree import STRtree
from tqdm import tqdm

In [198]:
# Function to merge many shapefile and point files
import os
import pandas as pd
import geopandas as gpd
#
def merge_shapefiles(base_dir, folders, shapefiles, output_filename="Merged.shp"):
    """
    Merge shapefiles given separate lists of folder names and shapefile names.
    Parameters:
        base_dir (str): Path to the base directory containing subdirectories.
        folders (list of str): List of subdirectory names.
        shapefiles (list of str): List of shapefile names inside corresponding subdirectories.
        output_filename (str): Name of the output merged shapefile (default "Merged.shp").
    Returns:
        str: Path to the saved merged shapefile.
    
    Raises:
        ValueError: If folders and shapefiles list lengths don't match.
    """
    if len(folders) != len(shapefiles):
        raise ValueError("folders and shapefiles lists must have the same length.")
    
    gdf_list = []

    for folder, shp_name in zip(folders, shapefiles):
        shp_path = os.path.join(base_dir, folder, shp_name)
        if os.path.exists(shp_path):
            gdf = gpd.read_file(shp_path)
            gdf_list.append(gdf)
        else:
            print(f"Warning: {shp_path} not found and will be skipped.")

    if not gdf_list:
        raise ValueError("No shapefiles found to merge.")

    merged_gdf = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True))

    output_path = os.path.join(base_dir, output_filename)
    merged_gdf.to_file(output_path)

    print(f"Merged shapefile saved to {output_path}")
    return output_path

In [ ]:
import geopandas as gpd

# Define base directory, subdirectory, input shapefile and output shapefile
base_dir = r'D:\Zelalem\WSC' 
folders = [f"MDA_ADP_{i:02d}" for i in range(1, 12)]
shapefiles = [f"MDA_ADP_{i:02d}_Station.shp" for i in range(1, 12)]
output_filename = "Merged_MDA_ADP_Stations.shp"
merge_shapefiles(base_dir, folders, shapefiles, output_filename)

# Define base directory, subdirectory, input shapefile and output shapefile
base_dir = r'D:\Zelalem\WSC'
folders = [f"MDA_ADP_{i:02d}" for i in range(1, 12)]
shapefiles = [f"MDA_ADP_{i:02d}_DrainageBasin_BassinDeDrainage.shp" for i in range(1, 12)]
output_filename = "Merged_MDA_ADP_DrainageBasin_BassinDeDrainage.shp"
merge_shapefiles(base_dir, folders, shapefiles, output_filename)

# The result will have one merged polygon per unique 'ID'
shape_path = r'D:\Zelalem\WSC\Merged_MDA_ADP_DrainageBasin_BassinDeDrainage.shp'
shape = gpd.read_file(shape_path)
# Dissolve polygons by 'ID' column
merged_shape = shape.dissolve(by='StationNum')
# Apply unary_union to each geometry to force full merging
merged_shape['geometry'] = merged_shape.geometry.apply(
    lambda geom: unary_union(geom.geoms) if geom.geom_type == 'MultiPolygon' else geom
)
# Reset index if needed
merged_shape = merged_shape.reset_index()
# Save the file
shape_path = r'D:\Zelalem\WSC\WSC_MDA_ADP_GaugedDrainageBasins.shp'
merged_shape.to_file(shape_path)

In [4]:
## REQ-0: Function to subset one geofabric using another

import geopandas as gpd
from shapely.geometry import Polygon
from shapely.strtree import STRtree

def subset_within_large_polygon_optimized(
    input_shapefile,
    big_polygon_shapefile: str
):
    """
    Efficiently subsets geometries that are strictly within a large polygon using spatial index.

    Parameters:
        input_shapefile (str): Path to shapefile with features to be subset.
        big_polygon_shapefile (str): Path to shapefile with one large polygon.
        output_shapefile (str): Path to save the output.
    """
    gdf = input_shapefile
    big_gdf = gpd.read_file(big_polygon_shapefile)
    if len(big_gdf) != 1:
        raise ValueError("The big polygon shapefile must contain exactly one feature.")
    big_polygon = big_gdf.geometry.iloc[0]
    if not isinstance(big_polygon, Polygon):
        raise TypeError("The geometry must be a Polygon.")

    # Ensure CRS match
    if gdf.crs != big_gdf.crs:
        if verbose: print("🔁 Reprojecting input to match big polygon CRS...")
        gdf = gdf.to_crs(big_gdf.crs)

    spatial_index = gdf.sindex
    candidate_idx = list(spatial_index.query(big_polygon, predicate='intersects'))
    candidates = gdf.iloc[candidate_idx]

    # Final exact match
    exact_matches = candidates[candidates.geometry.within(big_polygon)].copy()

    return exact_matches

# === Example Usage ===
# subset_within_large_polygon_optimized(
#     input_shapefile=input_shapefile,
#     big_polygon_shapefile=r"D:\study_domain\CanTrans_BasinBoundary_MERIT_withBuffer.shp",
# )

In [3]:
## REQ-1-Final: Function to extract COMID from geofabric using another basin shapefile

import geopandas as gpd
import pandas as pd
import numpy as np
from tqdm import tqdm

def match_basins_to_rivers(
    input_river_shapefile: str,
    input_basin_shapefile: str,
    gauged_basin_shapefile: str,
    output_matched_basin: str,
    output_matched_merit: str,
    area_field: str = 'Area_km2'
):
    print("📥 Loading shapefiles...")
    river_network = gpd.read_file(input_river_shapefile)
    basin_shape = gpd.read_file(input_basin_shapefile)
    gauged_basin = gpd.read_file(gauged_basin_shapefile)

    target_crs = gauged_basin.crs
    river_network = river_network.to_crs(target_crs)
    basin_shape = basin_shape.to_crs(target_crs)

    print("📍 Converting river lines to midpoints...")
    river_points = river_network.copy()
    # Make the point at aroound the start of the river 
    river_points['geometry'] = river_points.geometry.interpolate(0.99, normalized=True)

    print("🔁 Spatial join: Rivers inside gauged basins...")
    river_in_basin = gpd.sjoin(
        river_points[['COMID', 'geometry']],
        gauged_basin[['geometry']],
        how='left',
        predicate='within'
    )

    basins_with_rivers = river_in_basin['index_right'].dropna().unique()
    gauged_basin = gauged_basin.copy()
    gauged_basin['has_river'] = gauged_basin.index.isin(basins_with_rivers)

    with_river_basins = gauged_basin[gauged_basin['has_river']].copy()
    no_river_basins = gauged_basin[~gauged_basin['has_river']].copy()

    print(f"✅ {len(with_river_basins)} basins with river intersection (Case 1 & 3)")
    print(f"✅ {len(no_river_basins)} basins without river intersection (Case 2)")

    # === Case 2: No intersecting rivers ===
    no_river_basins['rep_point'] = no_river_basins.geometry.representative_point()
    rep_gdf = gpd.GeoDataFrame(no_river_basins[['rep_point']], geometry='rep_point', crs=no_river_basins.crs)

    merit_join = gpd.sjoin(rep_gdf, basin_shape[['COMID', 'geometry']], how='left', predicate='within')
    no_river_basins.loc[:, 'COMID'] = merit_join['COMID'].values

    fallback_mask = no_river_basins['COMID'].isna()
    if fallback_mask.any():
        print(f"⚠️ {fallback_mask.sum()} basins could not be matched — assigning nearest river COMID...")
        river_sindex = river_points.sindex
        fallback_points = no_river_basins.loc[fallback_mask, 'rep_point']
        fallback_comids = []

        for pt in tqdm(fallback_points, desc="Finding nearest river", dynamic_ncols=True):
            nearest_idx = list(river_sindex.nearest(pt, 1))[0]
            fallback_comids.append(river_points.iloc[nearest_idx]['COMID'])

        no_river_basins.loc[fallback_mask, 'COMID'] = fallback_comids

    # === Case 1 & 3: Rivers intersect ===
    print("📌 Processing intersecting rivers (Case 1 & 3)...")

    with_river_basins = with_river_basins.copy()
    with_river_basins['basin_index'] = with_river_basins.index
    intersections = gpd.sjoin(
        river_points[['COMID', 'geometry']],
        with_river_basins[['geometry', 'basin_index', area_field]],
        how='inner',
        predicate='within'
    )

    intersections = intersections.merge(
        river_network[['COMID', 'NextDownID', 'uparea']],
        on='COMID', how='left'
    )

    next_uparea_df = river_network[['COMID', 'uparea']].rename(columns={
        'COMID': 'NextDownID', 'uparea': 'next_uparea'
    })

    intersections = intersections.merge(
        next_uparea_df,
        on='NextDownID',
        how='left'
    )

    intersections['area_diff'] = (intersections['uparea'] - intersections[area_field]).abs()
    intersections['next_area_diff'] = (intersections['next_uparea'] - intersections[area_field]).abs()

    print("🔁 Tracing downstream for best outlet COMID...")

    best_comids = []
    for basin_id, group in tqdm(intersections.groupby('basin_index'), desc="Tracing outlets", dynamic_ncols=True):
        # Start from river with minimum area_diff (closest match to gauged basin area)
        current_row = group.loc[group['area_diff'].idxmin()].copy()
        visited = set()

        while True:
            current_comid = current_row['COMID']
            next_comid = current_row['NextDownID']
            visited.add(current_comid)

            # Stop if downstream COMID is missing or has been visited already (loop prevention)
            if pd.isna(next_comid) or next_comid in visited:
                break

            # Continue downstream if next_comid still inside basin
            if next_comid in group['COMID'].values:
                current_row = group[group['COMID'] == next_comid].iloc[0].copy()
            else:
                # NextDownID is outside basin — exit loop, current_row is outlet inside basin
                break

        # After exiting basin, final check — if downstream (outside basin) is closer or equal in area_diff, use that instead
        best_comid = current_row['COMID']
        next_diff = current_row.get('next_area_diff', np.inf)
        curr_diff = current_row.get('area_diff', np.inf)
        if pd.notna(next_diff) and next_diff <= curr_diff:
            best_comid = current_row['NextDownID']

        best_comids.append((basin_id, best_comid))

    best_df = pd.DataFrame(best_comids, columns=['basin_index', 'COMID'])
    with_river_basins = with_river_basins.merge(best_df, on='basin_index', how='left')

    # === Merge all matched ===
    all_matched = pd.concat([with_river_basins, no_river_basins])
    all_matched = gpd.GeoDataFrame(all_matched, crs=target_crs)
    all_matched = all_matched[all_matched['COMID'].notna()].copy()
    all_matched['COMID'] = all_matched['COMID'].astype(int)

    # === Enrich with attributes ===
    all_matched = all_matched.merge(
        basin_shape[['COMID', 'unitarea']],
        on='COMID', how='left'
    ).merge(
        river_network[['COMID', 'uparea', 'hillslope']],
        on='COMID', how='left'
    )

    # Fix hillslope rivers
    mask = all_matched['hillslope'] == 1
    all_matched.loc[mask, 'uparea'] = all_matched.loc[mask, 'unitarea']

    all_matched['DA_Diff'] = ((all_matched['uparea'] - all_matched[area_field]) / all_matched[area_field]) * 100

    # === Save outputs ===
    if 'rep_point' in all_matched.columns:
        all_matched = all_matched.drop(columns=['rep_point'])
    all_matched = all_matched.set_geometry('geometry')
    all_matched.to_file(output_matched_basin, driver="GPKG")

    #matched_merit = basin_shape[basin_shape['COMID'].isin(all_matched['COMID'])]
    #matched_merit = matched_merit.merge(river_network[['COMID', 'uparea']], on='COMID', how='left')
    #matched_merit.to_file(output_matched_merit, driver="GPKG")

    print("✅ Matching complete.")


In [200]:
# The function only replaces missing values
# It does NOT overwrite existing values, even if reference has different data
# Missing means any of: NaN, None, <NA>, '', 'Nodata', ' '

import geopandas as gpd
import pandas as pd
import numpy as np

def update_rows_vectorized(df, reference_df, key_col, cols_to_update, overwrite_existing=False):
    """
    Update df using reference_df.

    Parameters:
    ---------------------
    overwrite_existing : bool
        - False (default): update ONLY when main has missing values.
        - True: replace main values whenever reference has a valid value.

    Rules for validity:
    - Missing in main or reference includes: NaN, <NA>, None, '', 'Nodata', ' '.
    """

    df = df.copy()

    # Merge to bring reference columns
    merged = df.merge(
        reference_df[[key_col] + cols_to_update],
        on=key_col,
        suffixes=('', '_ref'),
        how='left'
    )

    def is_missing(series):
        """Detect all forms of missing data."""
        s = series.astype("object")
        return s.isna() | s.isin(['', ' ', 'Nodata', '<NA>', None])

    for col in cols_to_update:
        current = merged[col]
        ref = merged[col + '_ref']

        main_missing = is_missing(current)
        ref_valid   = ~is_missing(ref)

        if overwrite_existing:
            # Replace whenever reference has a valid value
            mask_update = ref_valid
        else:
            # Replace only when main is missing AND reference valid
            mask_update = main_missing & ref_valid

        merged[col] = np.where(mask_update, ref, current)

        # Drop the temporary reference column
        merged.drop(columns=[col + '_ref'], inplace=True)

        # Try restoring numeric dtype
        try:
            merged[col] = pd.to_numeric(merged[col])
        except Exception:
            pass

    return merged

In [ ]:
################ Mapping Guaged basin in the MERIT Hydro-Basin  ################################################
# Case:1 - Map WSC basins in MERIT-hydro basin
match_basins_to_rivers(
    input_river_shapefile=r'D:\Zelalem\MERIT\Orig_MERIT_CanTrans_rivers.shp',
    input_basin_shapefile=r'D:\Zelalem\MERIT\Orig_MERIT_CanTrans_subbasins.shp',
    gauged_basin_shapefile=r'D:\Zelalem\WSC\Merged_MDA_ADP_DrainageBasin_BassinDeDrainage.shp',
    output_matched_basin=r'D:\Zelalem\WSC\WSC_COMID_All_basins_plus.gpkg',
    output_matched_merit=r'D:\Zelalem\WSC\WSC_Stations_MERIT_COMID.gpkg',
    area_field="Area_km2"
)

# Case:2 - Map CAMEL-SPAT basins in MERIT-hydro basin
## Load the shapefile
input_path = r'D:\Zelalem\CamelSpat1698\cantrans_merged_lumped_outlines.shp'
output_path = r'D:\Zelalem\CamelSpat1698\reproj_cantrans_merged_lumped_outlines.shp'
gdf = gpd.read_file(input_path)

## Define North America Albers projection (manual PROJ string)
north_america_albers = (
    "+proj=aea +lat_1=20 +lat_2=60 +lat_0=40 "
    "+lon_0=-96 +x_0=0 +y_0=0 "
    "+datum=NAD83 +units=m +no_defs"
)
## Reproject
gdf_proj = gdf.to_crs(north_america_albers)
gdf_proj.to_file(output_path)

## Find the COMID of the outlet of the basin
match_basins_to_rivers(
    input_river_shapefile=r'D:\Zelalem\MERIT\Orig_MERIT_CanTrans_rivers.shp',
    input_basin_shapefile=r'D:\Zelalem\MERIT\Orig_MERIT_CanTrans_subbasins.shp',
    gauged_basin_shapefile=output_path,
    output_matched_basin=r'D:\Zelalem\CamelSpat1698\camels-spat_COMID_All_basins_plus.gpkg',
    output_matched_merit=r'D:\Zelalem\CamelSpat1698\camels-spat_Stations_MERIT_COMID.gpkg',
    area_field="Basin_area"
)

# Case3: - Map USGU-GAGE-II basins in MERIT-hydro basin
match_basins_to_rivers(
    input_river_shapefile=r'D:\Zelalem\MERIT\Orig_MERIT_CanTrans_rivers.shp',
    input_basin_shapefile=r'D:\Zelalem\MERIT\Orig_MERIT_CanTrans_subbasins.shp',
    gauged_basin_shapefile=r'D:\Zelalem\GAGES-II\cantrans_merged_shapefile.shp',
    output_matched_basin=r'D:\Zelalem\GAGES-II\gagesii_COMID_All_basins_plus.gpkg',
    output_matched_merit=r'D:\Zelalem\GAGES-II\gagesii_Stations_MERIT_COMID.gpkg',
    area_field="Area_km2"
)

In [41]:
# Case:4 - Map CLRH_NALRRP basins in MERIT-hydro basin
gpkg_path = r'D:\Zelalem\NALRRPv2p1\nalrrpv2p1_hydrofabric.gpkg'
nalrrp_gauges_basin = gpd.read_file(gpkg_path, layer="finalcat_info_v2_1")
nalrrp_gauges_basin["Obs_NM"] = nalrrp_gauges_basin["Obs_NM"].astype(str)

# Read the study domain shapefile and prepare the Outlets
CanTrans = gpd.read_file(r'D:\Zelalem\WSC\CanTrans_MERIT_StudyDomain.shp')
CanTrans = CanTrans.rename(columns={'COMID': 'ID'})

# Ensure CRS match
if nalrrp_gauges_basin.crs is not None:
    # Reproject CanTrans to cat's CRS
    CanTrans = CanTrans.to_crs(nalrrp_gauges_basin.crs)
# Confirm the new CRS
# print(f'CanTrans CRS after projection: {CanTrans.crs}')

# Compute interior centroids
basins = nalrrp_gauges_basin.copy()
basins['interior_centroid'] = basins.geometry.apply(lambda geom: geom.representative_point())
# Create GeoDataFrame of centroids with COMID
centroid_gdf = gpd.GeoDataFrame(
    basins[['SubId']].copy(),
    geometry=basins['interior_centroid'],
    crs=basins.crs
)
# Spatial join: keep only centroids within CanTrans polygons
centroids_within_target = gpd.sjoin(centroid_gdf, CanTrans, predicate='within', how='inner')
# Extract unique COMIDs
matching_comids = centroids_within_target['SubId'].unique().tolist()
# Filter cat GeoDataFrame using matching_comids
nalrrp_gauges_basin = nalrrp_gauges_basin[nalrrp_gauges_basin['SubId'].isin(matching_comids)]
nalrrp_gauges_basin = nalrrp_gauges_basin[nalrrp_gauges_basin['SRC_obs'] == 'US']
nalrrp_gauges_basin = nalrrp_gauges_basin[nalrrp_gauges_basin['Has_Gauge'] == 1]
nalrrp_gauges_basin = nalrrp_gauges_basin[~nalrrp_gauges_basin['Obs_NM'].str.contains('BORDER', na=False)]
# Read clrh_gauges_basin
gpkg_path = r'D:\Zelalem\CLRH_Basin\merged_finalcat_info_v1-0.gpkg'
clrh_gauges_basin = gpd.read_file(gpkg_path, layer="merged_finalcat_info_v1-0")
clrh_gauges_basin["Obs_NM"] = clrh_gauges_basin["Obs_NM"].astype(str)
clrh_gauges_basin = clrh_gauges_basin[clrh_gauges_basin['Has_POI'] == 1]
clrh_gauges_basin = clrh_gauges_basin[~clrh_gauges_basin['Obs_NM'].str.contains('BORDER', na=False)]
# Reproject nalrrp_gauges_basin to match clrh_gauges_basin
nalrrp_gauges_basin = nalrrp_gauges_basin.to_crs(clrh_gauges_basin.crs)
# Step 1: Find SubIds in clrh
clrh_subids = set(clrh_gauges_basin['Obs_NM'])
# Step 2: Filter nalrrp to exclude those already in clrh
nalrrp_unique = nalrrp_gauges_basin[~nalrrp_gauges_basin['Obs_NM'].isin(clrh_subids)]
# Step 3: Concatenate clrh + non-duplicate nalrrp
clrh_nalrrp = pd.concat([clrh_gauges_basin, nalrrp_unique], ignore_index=True)
# Ensure it's still a GeoDataFrame (if geometry is present)
if isinstance(clrh_nalrrp, gpd.GeoDataFrame):
    clrh_nalrrp = gpd.GeoDataFrame(clrh_nalrrp, geometry='geometry', crs=clrh_gauges_basin.crs)
# save the combined 
gpkg_path = r'D:\Zelalem\CLRH_Basin\clrh_nalrrp.gpkg'
clrh_nalrrp['DrainArea'] = clrh_nalrrp['DrainArea'] / 1_000_000
clrh_nalrrp.to_file(gpkg_path, driver='GPKG')

## CLRH & NALRRP
match_basins_to_rivers(
    input_river_shapefile=r'D:\Zelalem\MERIT\Orig_MERIT_CanTrans_rivers.shp',
    input_basin_shapefile=r'D:\Zelalem\MERIT\Orig_MERIT_CanTrans_subbasins.shp',
    gauged_basin_shapefile=r'D:\Zelalem\CLRH_Basin\clrh_nalrrp.gpkg',
    output_matched_basin=r'D:\Zelalem\CLRH_Basin\clrh_nalrrp_COMID_All_basins_plus.gpkg',
    output_matched_merit=r'D:\Zelalem\CLRH_Basin\clrh_nalrrp_Stations_MERIT_COMID.gpkg',
    area_field="DrainArea"
)

📥 Loading shapefiles...
📍 Converting river lines to midpoints...
🔁 Spatial join: Rivers inside gauged basins...
✅ 3362 basins with river intersection (Case 1 & 3)
✅ 8941 basins without river intersection (Case 2)
⚠️ 1 basins could not be matched — assigning nearest river COMID...


Finding nearest river: 100%|██████████| 1/1 [00:00<?, ?it/s]

📌 Processing intersecting rivers (Case 1 & 3)...


🔁 Tracing downstream for best outlet COMID...


Tracing outlets: 100%|██████████| 3362/3362 [00:00<00:00, 6428.00it/s]


✅ Matching complete.


In [201]:
# Combine the outlet of all stations COMID
wsc = gpd.read_file(r'D:\Zelalem\WSC\WSC_COMID_All_basins_plus.gpkg')
camelspat = gpd.read_file(r'D:\Zelalem\CamelSpat1698\camels-spat_COMID_All_basins_plus.gpkg')
gagesii = gpd.read_file(r'D:\Zelalem\GAGES-II\gagesii_COMID_All_basins_plus.gpkg')
clrh_nalrrp = gpd.read_file(r'D:\Zelalem\CLRH_Basin\clrh_nalrrp_COMID_All_basins_plus.gpkg')

# Step 1: Standardize StationID
wsc['StationID'] = wsc['StationNum'].astype(str)
gagesii['StationID'] = gagesii['GAGE_ID'].astype(str)
camelspat['StationID'] = camelspat['Station_id'].astype(str)
clrh_nalrrp['StationID'] = clrh_nalrrp['Obs_NM'].astype(str)

# Step 2: Rename COMID columns for clarity
wsc = wsc[['StationID', 'COMID']].rename(columns={'COMID': 'COMID_WSC'})
gagesii = gagesii[['StationID', 'COMID']].rename(columns={'COMID': 'COMID_GAGE'})
camelspat = camelspat[['StationID', 'COMID']].rename(columns={'COMID': 'COMID_CAMELS'})
# Step 1: Split concatenated StationIDs by '&' and explode into separate rows
clrh_nalrrp_expanded = (
    clrh_nalrrp
    .assign(StationID = clrh_nalrrp['StationID'].str.split('&'))  # Split into list
    .explode('StationID')  # One row per StationID
    .reset_index(drop=True)
)
# Step 2: Keep only relevant columns
clrh_nalrrp_expanded = clrh_nalrrp_expanded[['StationID','COMID']].rename(columns={'COMID':'COMID_CLRHNA'})
# Combine all shapefiles (no dissolve!)
combined_df = pd.concat([wsc, gagesii, camelspat, clrh_nalrrp_expanded], ignore_index=True)
# Keep only unique StationIDs
combined_df = combined_df.drop_duplicates(subset='StationID', keep='first').reset_index(drop=True)
# Drop a column named 'column_to_drop'
combined_df = combined_df.drop(columns=['COMID_WSC', 'COMID_GAGE', 'COMID_CAMELS', 'COMID_CLRHNA'])

# Step 4: Merge COMIDs
combined_df = combined_df.merge(wsc, on='StationID', how='left')
combined_df = combined_df.merge(gagesii, on='StationID', how='left')
combined_df = combined_df.merge(camelspat, on='StationID', how='left')
combined_df = combined_df.merge(clrh_nalrrp_expanded, on='StationID', how='left')

# Step 5: Final output
combined_df = combined_df.sort_values(by='StationID').reset_index(drop=True)

# Replacing COMID based on the following order WSC--GAGEII--CAMELS--CLRH
combined_df['COMID'] = combined_df['COMID_WSC']
combined_df['COMID'] = combined_df['COMID'].fillna(combined_df['COMID_GAGE'])
combined_df['COMID'] = combined_df['COMID'].fillna(combined_df['COMID_CAMELS'])
combined_df['COMID'] = combined_df['COMID'].fillna(combined_df['COMID_CLRHNA'])
#
combined_df = combined_df.rename(columns={"StationID": "Obs_NM"})
combined_df.to_csv(r'D:\Zelalem\WSC\all_combined_station_comid.csv', index=False)

#compare_cols = ['COMID_WSC', 'COMID_CAMELS']
#filtered_df = combined_df[combined_df[compare_cols].apply(lambda row: row.dropna().nunique() > 1, axis=1)]
#compare_cols = ['COMID_WSC', 'COMID_CLRHNA']
#filtered_df = combined_df[combined_df[compare_cols].apply(lambda row: row.dropna().nunique() > 1, axis=1)]

In [202]:
# Combine stations from available sources into one
import geopandas as gpd
import pandas as pd
import numpy as np

# Load the polygon shapefiles
input_basin = 'D:\\Zelalem\\MERIT\\Newagg\\sorted_agg_MERIT_CanTrans_subbasins.shp'
polygons = gpd.read_file(input_basin)

# Load CLRH as a base shapefile
gdf1 = gpd.read_file(r'D:\Zelalem\CLRH_Basin\merged_poi_v1-0.gpkg')
gdf1["Obs_NM"] = gdf1["Obs_NM"].astype(str)
gdf1 = gdf1[~gdf1['Obs_NM'].str.contains('BORDER', na=False)]
gdf1.loc[gdf1['SRC_obs'].str.strip() == 'PROVINCIAL', 'SRC_obs'] = 'WSC'
# Change DA less or equal zero into missing value
gdf1.loc[gdf1['DA_Obs'] <= 0, 'DA_Obs'] = np.nan

# Load NALRRPv2p1 point shpefile
gdf2 = gpd.read_file(r'D:\Zelalem\NALRRPv2p1\nalrrpv2p1_hydrofabric.gpkg', layer="obs_gauges_v2_1_clip")
gdf2["Obs_NM"] = gdf2["Obs_NM"].astype(str)
gdf2['SRC_obs'] = gdf2['SRC_obs'].str.upper().replace({'US': 'USGS', 'CA': 'WSC'})
gdf2['data_tp'] = 'Nodata'
gdf2['Gauge_nm'] = 'Nodata'
# Add numeric longitude and latitude from geometry
gdf2['Lon'] = gdf2.geometry.x
gdf2['Lat'] = gdf2.geometry.y
# Change DA less or equal zero into missing value
gdf2.loc[gdf2['DA_Obs'] <= 0, 'DA_Obs'] = np.nan
# Load USGS shapefile and rename columns
gdf3 = gpd.read_file(r'D:\Zelalem\SWIM_gage_loc\SWIM_gage_loc.shp')
gdf3 = gdf3.rename(columns={
    'Gage_no': 'Obs_NM',
    'Gage_type': 'data_tp',
    'Gage_name': 'Gauge_nm',
    'SqKm_nwis': 'DA_Obs',
    'Lat_nwis': 'Lat',
    'Long_nwis': 'Lon'
})
# Make Obs_NM unique by keeping the first occurrence
gdf3["Obs_NM"] = gdf3["Obs_NM"].astype(str)
gdf3 = gdf3.groupby('Obs_NM', as_index=False).first()
# Change DA less or equal zero into missing value
gdf3.loc[gdf3['DA_Obs'] <= 0, 'DA_Obs'] = np.nan
# Rename to Discharge and Level 
gdf3['data_tp'] = gdf3['data_tp'].str.lower().replace({'contgage': 'Discharge', 'creststage': 'Level'})
# Define columns to update
cols = ['Gauge_nm', 'DA_Obs', 'data_tp', 'Lat', 'Lon']
# Apply the vectorized update
gdf2 = update_rows_vectorized(gdf2, gdf3, key_col='Obs_NM', cols_to_update=cols, overwrite_existing=True)
# Ensure CRS matches gdf1
if gdf2.crs != gdf1.crs:
    gdf2 = gdf2.to_crs(gdf1.crs)
# Step 1: Create sets of existing IDs
ids_gdf1 = set(gdf1['Obs_NM'])
ids_gdf2 = set(gdf2['Obs_NM'])
# Step 2: Get only new rows from gdf2 where Obs_NM is not already in gdf1
missing_gdf2 = gdf2[~gdf2['Obs_NM'].isin(ids_gdf1)]
# Step 3: Add missing gdf2 rows to gdf1
gdf1 = pd.concat([gdf1, missing_gdf2], ignore_index=True)
# Load WSC shapefile and tag source
gdf4 = gpd.read_file(r'D:\Zelalem\WSC\Hydat_Stations.shp')
gdf4['SRC_obs'] = 'WSC'
# First, rename StationNum → Obs_NM to match gdf1
gdf4 = gdf4.rename(columns={
    'STATION_NU': 'Obs_NM',
    'STATION_NA': 'Gauge_nm',
    'LATITUDE': 'Lat',
    'LONGITUDE': 'Lon',
    'DRAINAGE_A': 'DA_Obs'
})
# Update data type of the measurment
# Ensure 'data_tp' exists and is string type
gdf4['data_tp'] = pd.NA  
# Convert HasFlow/HasLevel to integers
gdf4['HasFlow'] = gdf4['HasFlow'].astype(int)
gdf4['HasLevel'] = gdf4['HasLevel'].astype(int)
# Case 1: Discharge only
gdf4.loc[(gdf4['HasFlow'] == -1) & (gdf4['HasLevel'] != -1), 'data_tp'] = 'Discharge'
# Case 2: Level only
gdf4.loc[(gdf4['HasFlow'] != -1) & (gdf4['HasLevel'] == -1), 'data_tp'] = 'Level'
# Case 3: Both Discharge and Level
gdf4.loc[(gdf4['HasFlow'] == -1) & (gdf4['HasLevel'] == -1), 'data_tp'] = 'Discharge/Level'
# Case 4: Neither (optional)
gdf4.loc[(gdf4['HasFlow'] != -1) & (gdf4['HasLevel'] != -1), 'data_tp'] = pd.NA
# Change DA less or equal zero into missing value
gdf4.loc[gdf4['DA_Obs'] <= 0, 'DA_Obs'] = np.nan
# Load provical data
df1 = pd.read_csv(r'D:\Zelalem\QuébecGovernmentHydrometricNetwork\stations_hydrometriques.csv')
df1['type'] = df1['type'].str.lower().replace({
    'débit': 'Discharge',
    'niveau': 'Level',
    'débit et niveau': 'Discharge/Level'
})
df1['etat'] = df1['etat'].str.lower().replace({
    'fermée': 'D',
    'ouverte': 'A'
})
df1['regime'] = df1['regime'].str.lower().replace({
    'non influencé': 'FALSE',
    'influencé': 'TRUE',
    'à déterminer': np.nan,
    'peu influencé': 'TRUE'
})
df1 = df1.rename(columns={
    'no': 'Obs_NM',
    'nom': 'Gauge_nm',
    'superficie': 'DA_Obs',
    'type': 'data_tp',
    'latitude': 'Lat',
    'longitude': 'Lon',
    'regime': 'IsRegulate',
    'etat': 'HYD_STATUS'
})
df1['SRC_obs'] = 'WSC'

# Define columns to update
cols = ['Gauge_nm', 'DA_Obs', 'data_tp', 'Lat', 'Lon', 
        'IsRegulate', 'HYD_STATUS', 'SRC_obs']
# Change DA less or equal zero into missing value
df1.loc[df1['DA_Obs'] <= 0, 'DA_Obs'] = np.nan
# Update the values from provical data
gdf4 = update_rows_vectorized(gdf4, df1, key_col='Obs_NM', cols_to_update=cols)
# Load the point shapefile (WSC shapefile)
gdf5 = gpd.read_file(r'D:\Zelalem\WSC\Merged_MDA_ADP_DrainageBasin_BassinDeDrainage.shp')
gdf5 = gdf5.rename(columns={'StationNum': 'Obs_NM', 'NameNom': 'Gauge_nm', 'Area_km2': 'DA_Obs', 'Status': 'HYD_STATUS'})
gdf5['HYD_STATUS'] = gdf5['HYD_STATUS'].replace({'discontinued': 'D', 'active': 'A'})
# Create a mapping from gdf5
cols = ['Gauge_nm', 'HYD_STATUS', 'DA_Obs']
# Change DA less or equal zero into missing value
gdf5.loc[gdf5['DA_Obs'] <= 0, 'DA_Obs'] = np.nan
# Apply the values from the WSC new shapefile
gdf4 = update_rows_vectorized(gdf4, gdf5, key_col='Obs_NM', cols_to_update=cols, overwrite_existing=True)
# Now combine the two geodataframe: gdf1 and gdf4
# Ensure CRS matches between gdf1 and gdf4
if gdf4.crs != gdf1.crs:
    gdf4 = gdf4.to_crs(gdf1.crs)
# Step 1: Create sets of existing IDs
ids_gdf1 = set(gdf1['Obs_NM'])
# Get only new rows from gdf4 where Obs_NM is not already in gdf1
missing_gdf4 = gdf4[~gdf4['Obs_NM'].isin(ids_gdf1)]
# Append missing rows from gdf4
gdf_combined = pd.concat([
    gdf1,
    missing_gdf4[['Obs_NM', 'Gauge_nm', 'DA_Obs', 'SRC_obs', 'data_tp', 'Lat', 'Lon', 'geometry']]
], ignore_index=True)
# Ensure final result is a valid GeoDataFrame with correct geometry and CRS
gdf_combined = gpd.GeoDataFrame(gdf_combined, geometry='geometry', crs=gdf1.crs)
gdf_combined = pd.merge(gdf_combined, gdf4[['Obs_NM', 'RHBN', 'HYD_STATUS', 'IsRegulate']], on='Obs_NM', how='left')
# Apply the values from the WSC new shapefile
cols = ['Gauge_nm', 'HYD_STATUS', 'DA_Obs', 'RHBN', 'IsRegulate', 'data_tp', 'SRC_obs', 'Lat', 'Lon']
gdf_combined = update_rows_vectorized(gdf_combined, gdf4, key_col='Obs_NM', cols_to_update=cols, overwrite_existing=True)
# Add COMIDs from the above steps
gdf_combined = gdf_combined.merge(combined_df, on='Obs_NM', how='left')
# Ensure both layers use the same CRS
gdf_combined = gdf_combined.to_crs(polygons.crs)
# Filter points where COMID is NaN
points_nan_comids = gdf_combined[pd.isna(gdf_combined['COMID'])]
# Perform spatial join to assign polygon attributes to points
points_with_nan_comids = gpd.sjoin(points_nan_comids, polygons[['geometry', 'COMID']], how="left", predicate="intersects")
# Rename polygon COMID to avoid conflict
points_with_nan_comids = points_with_nan_comids.rename(columns={"COMID": "COMID_right"})
# Update original points GeoDataFrame only for rows with COMID == 0
gdf_combined.loc[points_nan_comids.index, 'COMID'] = points_with_nan_comids['COMID_right'].values
gdf_combined = gdf_combined[~gdf_combined['COMID'].isna()]
# Add drainage area based on the comid from the above steps
gdf_combined = gdf_combined.merge(polygons[['COMID', 'uparea', 'unitarea']], on='COMID', how='left')
gdf_combined.loc[gdf_combined['uparea'] <= 1e-6, 'uparea'] = gdf_combined.loc[gdf_combined['uparea'] <= 1e-6, 'unitarea']
gdf_combined['DrainArea'] = gdf_combined['uparea']  # copy 'uparea' to 'DrainArea'
gdf_combined = gdf_combined.drop(columns=['uparea', 'uparea', 'Id', 'DA_error', 'SSDA_NM', 'SDA_NM', 'Region', 'Use_region', 'Sub_Reg'])  # remove the old 'uparea' column
# Calculate the Drainage area difference in percentage
gdf_combined['DA_Diff'] = (gdf_combined['DrainArea'] - gdf_combined['DA_Obs']) / gdf_combined['DA_Obs'] * 100
# Define the desired column order
cols_order = ['Obs_NM', 'Gauge_nm', 'Lat', 'Lon', 'data_tp', 'HYD_STATUS',
              'SRC_obs', 'RHBN', 'IsRegulate', 'DA_Obs', 'DrainArea', 'DA_Diff', 
              'COMID_WSC', 'COMID_GAGE', 'COMID_CAMELS', 'COMID_CLRHNA', 'COMID', 'SubId', 'geometry'] 
# Reorder columns
gdf_combined = gdf_combined[cols_order]
# Remove duplicate rows
gdf_combined = gdf_combined.drop_duplicates(subset="Obs_NM", keep="first")
# Save to GeoPackage
gdf_combined.to_file(r'D:\Zelalem\WSC\merged-all-stations.gpkg', layer='points', driver="GPKG")

C:\Users\TesemmaZ\AppData\Local\anaconda3\envs\gismodule\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_read(
